# Day 3: FAISS Index + Visual Retrieval

**Goal**: Build FAISS index and do image-to-image retrieval with visual results.

**Done checkpoint**:
- FAISS index built and saved
- Query with any image returns top-10 visually similar results
- Results displayed as an image grid inline in Colab

**Runtime**: CPU is fine (FAISS flat index search is fast).

## 1. Mount Drive + Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_DIR = '/content/drive/MyDrive/Projects/VisualSearchSystem'
LOCAL_DATA_DIR = '/content/data'

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f'CWD: {os.getcwd()}')

In [ ]:
import zipfile
from pathlib import Path

# Images must be on local disk — paths in images.csv point to /content/data/raw/...
LOCAL_RAW_DIR = Path(LOCAL_DATA_DIR) / 'raw'
LOCAL_RAW_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR = LOCAL_RAW_DIR / 'images'

if IMAGES_DIR.exists() and any(IMAGES_DIR.iterdir()):
    print(f'Images on local disk: {len(list(IMAGES_DIR.glob("*.jpg"))):,} files.')
else:
    drive_zips = list((Path(PROJECT_DIR) / 'data' / 'raw').glob('*.zip'))
    if drive_zips:
        print('Extracting dataset from Drive zip to local disk...')
        with zipfile.ZipFile(drive_zips[0], 'r') as zf:
            zf.extractall(LOCAL_RAW_DIR)
        print(f'Done. {len(list(IMAGES_DIR.glob("*.jpg"))):,} images ready.')
    else:
        print('No zip found on Drive. Run Day 1 notebook first.')

In [ ]:
!pip install -q torch torchvision transformers faiss-cpu Pillow pandas numpy tqdm pyyaml
print('Done.')

## 2. Load Config

In [ ]:
import yaml
from pathlib import Path

with open(f'{PROJECT_DIR}/configs/config.yaml') as f:
    config = yaml.safe_load(f)

BASE  = Path(PROJECT_DIR)
LOCAL = Path(LOCAL_DATA_DIR)

# Images read from local disk (paths stored in images.csv point here)
config['paths']['raw_images']      = str(LOCAL / 'raw')
# All index/embedding files read from Drive (persistent outputs from Day 2)
config['paths']['metadata_csv']    = str(BASE / 'data/metadata/images.csv')
config['paths']['embeddings_path'] = str(BASE / 'data/embeddings/embeddings.npy')
config['paths']['id_map_path']     = str(BASE / 'data/embeddings/id_map.json')
config['paths']['index_path']      = str(BASE / 'data/embeddings/faiss.index')
config['model']['clip_model']      = 'openai/clip-vit-base-patch32'
config['model']['device']          = 'auto'  # CPU is fine for Day 3
print('Config ready.')

## 3. Build FAISS Index

In [ ]:
from pathlib import Path

index_path = Path(config['paths']['index_path'])

if index_path.exists():
    print(f'Index already exists at {index_path}. Skipping build.')
    print('Delete it and re-run to rebuild.')
else:
    from scripts.build_index import build_index
    build_index(config)
    print('Index built.')

## 4. Initialize Searcher

In [ ]:
from src.retrieval.search import Searcher

searcher = Searcher(config)
print(f'Searcher ready.')
print(f'  Index size:  {searcher.index.index.ntotal:,} vectors')
print(f'  Metadata:    {len(searcher.metadata):,} rows')

## 5. Search — Pick a Query Image

In [ ]:
import pandas as pd
import random
from pathlib import Path

# Pick a random image to use as query
df = pd.read_csv(config['paths']['metadata_csv'])
sample = df.sample(1).iloc[0]
QUERY_PATH = sample['path']
print(f'Query image:')
print(f'  ID:       {sample["image_id"]}')
print(f'  Category: {sample.get("category", "N/A")}')
print(f'  Product:  {sample.get("product_name", "N/A")}')
print(f'  Path:     {QUERY_PATH}')

In [ ]:
# Or pin a specific image ID for reproducibility:
# QUERY_ID = 1163  # change this to any valid image_id
# QUERY_PATH = df[df['image_id'] == QUERY_ID]['path'].values[0]

# Display query image
from PIL import Image
from IPython.display import display

img = Image.open(QUERY_PATH)
img = img.resize((224, 224))
print('Query image:')
display(img)

## 6. Run Visual Search

In [ ]:
results = searcher.search(QUERY_PATH, k=12)

print(f'Top {len(results)} results:')
for i, r in enumerate(results, 1):
    print(f'  {i:2}. [{r["image_id"]:6}] score={r["score"]:.4f}  {r.get("category","")}  {r.get("product_name","")[:40]}')

## 7. Display Results as Image Grid

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from pathlib import Path

def show_results(query_path, results, cols=4):
    n = len(results) + 1  # +1 for query
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3.5))
    axes = axes.flatten()

    # Query image
    ax = axes[0]
    ax.imshow(Image.open(query_path).resize((224, 224)))
    ax.set_title('QUERY', color='blue', fontweight='bold', fontsize=9)
    ax.axis('off')

    # Results
    for i, r in enumerate(results):
        ax = axes[i + 1]
        img_path = r.get('path')
        if img_path and Path(img_path).exists():
            ax.imshow(Image.open(img_path).resize((224, 224)))
        else:
            ax.text(0.5, 0.5, f'ID:{r["image_id"]}', ha='center', va='center')
        label = f"#{i+1} {r.get('category','')[:12]}\n{r.get('color','')[:10]}\n{r['score']:.3f}"
        ax.set_title(label, fontsize=7)
        ax.axis('off')

    for ax in axes[n:]:
        ax.axis('off')

    plt.suptitle('Visual Search Results', fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

show_results(QUERY_PATH, results)

## 8. Try Multiple Query Images

In [ ]:
# Run 3 random queries to get a feel for retrieval quality
for i in range(3):
    sample = df.sample(1).iloc[0]
    query_path = sample['path']
    results = searcher.search(query_path, k=8)
    print(f'\nQuery {i+1}: {sample.get("product_name", "")} | {sample.get("category", "")}')
    show_results(query_path, results, cols=4)

## ✅ Day 3 Done Checkpoint

In [ ]:
from pathlib import Path

assert Path(config['paths']['index_path']).exists(), 'FAISS index not found!'
test_results = searcher.search(QUERY_PATH, k=10)
assert len(test_results) == 10, f'Expected 10 results, got {len(test_results)}'
assert all('score' in r and 'image_id' in r for r in test_results)

print('Day 3 COMPLETE')
print(f'  FAISS index: {searcher.index.index.ntotal:,} vectors')
print(f'  Search test: {len(test_results)} results returned')
print(f'  Top score:   {test_results[0]["score"]:.4f}')